# Start Partition Influence Comparison

This notebook analyzes how the choice of start partition affects the quality and runtime of the local-search pipeline.

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

In [13]:
PLATEAU_STEPS = 4
PLATEAU_PIPELINE = ",".join(["move_plateau"] * PLATEAU_STEPS)

GRAPH_ORDER = ["powerlaw", "er"]

DATASET_ORDER = [
    "small sparse",
    "small dense",
    "large sparse",
    "large dense",
]

START_PARTITION_ORDER = [
    "singleton",
    "matching",
    "maximum_matching",
    "maximum_matching_edge_cover",
    "high_degree_first_matching",
    "high_degree_product_matching",
    "leiden_mdgp",
    "kapoce",
]

RESULTS_DIR = Path("../../results/experiment2/zero_gain_limit")
RAW_RESULTS_FILE = RESULTS_DIR / "raw_results.csv"

## Load data

In [14]:
raw = pd.read_csv(RAW_RESULTS_FILE)

raw["dataset_group"] = raw["size_class"].astype(str) + " " + raw["regime"].astype(str)

raw["graph_type"] = pd.Categorical(
    raw["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

raw["dataset_group"] = pd.Categorical(
    raw["dataset_group"],
    categories=DATASET_ORDER,
    ordered=True,
)

raw = raw[(raw["pipeline"] == PLATEAU_PIPELINE) & (raw["zero_gain_factor"] == 4)].copy()

raw["start_partition"] = pd.Categorical(
    raw["start_partition"],
    categories=START_PARTITION_ORDER,
    ordered=True,
)

## Solution quality

For every instance and start partition, only the run with the highest final solution quality is retained. The best result across all compared start partitions is used as the reference.

Relative solution quality is defined as

$
\frac{\text{best result across all start partitions}}
     {\text{result of the respective start partition}}.
$

A value of $1.0$ means that the start-partition matches the best result found on the same instance. Values greater than $1.0$ indicate the remaining quality gap.

In [15]:
best_run_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
    "start_partition",
]

best_runs = (
    raw
    .sort_values(["final_density", "ls_runtime", "run"], ascending=[False, True, True])
    .groupby(best_run_keys, observed=True, as_index=False)
    .head(1)
    .reset_index(drop=True)
)

In [16]:
instance_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
]

density_table = best_runs.pivot_table(
    index=instance_keys,
    columns="start_partition",
    values="final_density",
    observed=True,
)[START_PARTITION_ORDER]

best_per_instance = density_table.max(axis=1)
relative_to_best = density_table.rdiv(best_per_instance, axis=0)

relative_quality_summary = (
    relative_to_best
    .groupby(level=["graph_type", "dataset_group"])
    .mean()
    .stack()
    .rename("mean_relative_to_best")
    .reset_index()
)

relative_quality_summary

,graph_type,dataset_group,start_partition,mean_relative_to_best
0,powerlaw,small sparse,singleton,1.003132
1,powerlaw,small sparse,matching,1.002632
2,powerlaw,small sparse,maximum_matching,1.001936
3,powerlaw,small sparse,maximum_matching_edge_cover,1.002003
4,powerlaw,small sparse,high_degree_first_matching,1.002690
...,...,...,...,...
59,er,large dense,maximum_matching_edge_cover,1.010255
60,er,large dense,high_degree_first_matching,1.010402
61,er,large dense,high_degree_product_matching,1.010289
62,er,large dense,leiden_mdgp,1.006800


In [17]:
overall_relative_quality_summary = (
    relative_to_best
    .mean()
    .rename("mean_relative_to_best")
    .reset_index()
)

overall_relative_quality_summary

,start_partition,mean_relative_to_best
0,singleton,1.004715
1,matching,1.004580
2,maximum_matching,1.004227
3,maximum_matching_edge_cover,1.004244
4,high_degree_first_matching,1.004665
5,high_degree_product_matching,1.004621
6,leiden_mdgp,1.005928
7,kapoce,1.006204


## Runtime

The reported runtime is the total local-search runtime of all runs for one instance and operator, averaged over all instances in the corresponding dataset group.

In [18]:
runtime_per_instance = (
    raw
    .groupby(best_run_keys, observed=True, as_index=False)
    .agg(total_runtime=("ls_runtime", "sum"))
)

## Pairwise comparison with Maximum Matching

The selected start partition `maximum_matching` is compared directly with each alternative. The comparison reports how often Maximum Matching produces a better or worse solution and the relative difference in mean local-search runtime.

In [19]:
REFERENCE = "maximum_matching"

reference_quality = density_table[REFERENCE]

reference_runtime = (
    runtime_per_instance[runtime_per_instance["start_partition"] == REFERENCE]
    .set_index(instance_keys)["total_runtime"]
)

pairwise_rows = []

for other in START_PARTITION_ORDER:
    if other == REFERENCE:
        continue

    other_quality = density_table[other]

    other_runtime = (
        runtime_per_instance[runtime_per_instance["start_partition"] == other]
        .set_index(instance_keys)["total_runtime"]
    )

    pairwise_rows.append(
        {
            "opponent": other,
            "mm_better_rate": (reference_quality > other_quality).mean(),
            "mm_worse_rate": (reference_quality < other_quality).mean(),
            "runtime_difference": (other_runtime.mean() - reference_runtime.mean()) / reference_runtime.mean() * 100,
        }
    )

pairwise_summary = pd.DataFrame(pairwise_rows)

pairwise_summary["opponent"] = pd.Categorical(
    pairwise_summary["opponent"],
    categories=[start_partition for start_partition in START_PARTITION_ORDER if start_partition != REFERENCE],
    ordered=True,
)

pairwise_summary = (
    pairwise_summary
    .sort_values("opponent")
    .reset_index(drop=True)
)

pairwise_summary

,opponent,mm_better_rate,mm_worse_rate,runtime_difference
0,singleton,0.5525,0.3605,-3.783332
1,matching,0.5295,0.3770,-5.513374
2,maximum_matching_edge_cover,0.2100,0.2010,0.854656
3,high_degree_first_matching,0.5275,0.3735,-7.460101
4,high_degree_product_matching,0.5235,0.3755,-6.784904
5,leiden_mdgp,0.6040,0.3230,3.123139
6,kapoce,0.5410,0.4125,225.628603


## LaTeX helper functions

In [20]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor

def latex_start_partition(start_partition: str) -> str:
    return r"\texttt{" + start_partition.replace("_", r"\_") + "}"

def format_number(value: float, decimals: int) -> str:
    return f"{value:.{decimals}f}"

def format_percent(value: float, decimals: int = 1) -> str:
    return (rf"{truncate_number(100 * value, decimals):.{decimals}f}\,\%"
    )

def format_signed_number(value: float, decimals: int) -> str:
    return f"{value:+.{decimals}f}"

## Build LaTeX tables

In [27]:
def make_quality_latex_table(df: pd.DataFrame, graph_type: str, caption: str, label: str) -> str:
    graph_df = df[df["graph_type"] == graph_type].copy()

    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lp{5.0cm}r}",
        r"\toprule",
        r"Dataset & Initial partition & \shortstack{Mean relative\\solution quality} \\",
        r"\midrule",
    ]

    for dataset_index, dataset in enumerate(DATASET_ORDER):
        part = graph_df[graph_df["dataset_group"] == dataset].sort_values("start_partition")

        best_mean = part["mean_relative_to_best"].min()

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset}}}"
                if row_index == 0
                else ""
            )

            mean_quality = format_number(row.mean_relative_to_best, 6)

            if np.isclose(row.mean_relative_to_best, best_mean):
                mean_quality = rf"\textbf{{{mean_quality}}}"

            lines.append(
                f"{dataset_cell} "
                f"& {latex_start_partition(str(row.start_partition))} "
                f"& {mean_quality} "
                r"\\"
            )

        if dataset_index < len(DATASET_ORDER) - 1:
            lines.append(r"\cmidrule(l){1-3}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [28]:
powerlaw_quality_latex = make_quality_latex_table(
    relative_quality_summary,
    graph_type="powerlaw",
    caption=(
        "Comparison of the solution quality obtained from different initial partitions on Powerlaw instances."
    ),
    label="tab:start_partition_quality_powerlaw",
)

print(powerlaw_quality_latex)

\begin{table}[!htbp]
\centering
\caption{Comparison of the solution quality obtained from different initial partitions on Powerlaw instances.}
\label{tab:start_partition_quality_powerlaw}
\begin{tabular}{lp{5.0cm}r}
\toprule
Dataset & Initial partition & \shortstack{Mean relative\\solution quality} \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & 1.003132 \\
 & \texttt{matching} & 1.002632 \\
 & \texttt{maximum\_matching} & \textbf{1.001936} \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.002003 \\
 & \texttt{high\_degree\_first\_matching} & 1.002690 \\
 & \texttt{high\_degree\_product\_matching} & 1.002661 \\
 & \texttt{leiden\_mdgp} & 1.006005 \\
 & \texttt{kapoce} & 1.004331 \\
\cmidrule(l){1-3}
\multirow{8}{*}{small dense} & \texttt{singleton} & 1.004302 \\
 & \texttt{matching} & 1.004596 \\
 & \texttt{maximum\_matching} & \textbf{1.004232} \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.004296 \\
 & \texttt{high\_degree\_first\_matching} & 1.004601 \\
 & \textt

In [29]:
er_quality_latex = make_quality_latex_table(
    relative_quality_summary,
    graph_type="er",
    caption=(
        "Comparison of the solution quality obtained from different initial partitions on Erdős-Rényi instances."
    ),
    label="tab:start_partition_quality_er",
)

print(er_quality_latex)

\begin{table}[!htbp]
\centering
\caption{Comparison of the solution quality obtained from different initial partitions on Erdős-Rényi instances.}
\label{tab:start_partition_quality_er}
\begin{tabular}{lp{5.0cm}r}
\toprule
Dataset & Initial partition & \shortstack{Mean relative\\solution quality} \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & 1.004289 \\
 & \texttt{matching} & \textbf{1.003906} \\
 & \texttt{maximum\_matching} & 1.004661 \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.004575 \\
 & \texttt{high\_degree\_first\_matching} & 1.004324 \\
 & \texttt{high\_degree\_product\_matching} & 1.004368 \\
 & \texttt{leiden\_mdgp} & 1.005637 \\
 & \texttt{kapoce} & 1.009435 \\
\cmidrule(l){1-3}
\multirow{8}{*}{small dense} & \texttt{singleton} & \textbf{1.004848} \\
 & \texttt{matching} & 1.005325 \\
 & \texttt{maximum\_matching} & 1.005142 \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.005215 \\
 & \texttt{high\_degree\_first\_matching} & 1.004950 \\
 & \texttt{h

In [32]:
def make_pairwise_latex_table(pairwise_df: pd.DataFrame, quality_df: pd.DataFrame, caption: str, label: str) -> str:
    quality = quality_df.set_index("start_partition")["mean_relative_to_best"]

    lines = [
        r"\begin{table}[H]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{p{4.8cm}rrrr}",
        r"\toprule",
        r"Initial partition & \shortstack{Mean relative\\solution quality} & \shortstack{Maximum\\Matching\\better} & \shortstack{Maximum\\Matching\\worse} & \shortstack{Run time\\difference (\%)} \\",
        r"\midrule",
    ]

    for start_partition in START_PARTITION_ORDER:
        quality_value = format_number(quality[start_partition], 6)

        if start_partition == REFERENCE:
            quality_value = rf"\textbf{{{quality_value}}}"

            lines.append(
                f"{latex_start_partition(start_partition)} "
                f"& {quality_value} & -- & -- & -- \\\\"
            )
            continue

        row = pairwise_df[pairwise_df["opponent"] == start_partition].iloc[0]

        lines.append(
            f"{latex_start_partition(start_partition)} "
            f"& {quality_value} "
            f"& {format_percent(row['mm_better_rate'], 1)} "
            f"& {format_percent(row['mm_worse_rate'], 1)} "
            f"& {format_signed_number(row['runtime_difference'], 1)}"
            r"\,\% \\"
        )

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [33]:
pairwise_latex = make_pairwise_latex_table(
    pairwise_summary,
    overall_relative_quality_summary,
    caption=(
        "Pairwise comparison of \\texttt{maximum\\_matching} with the alternative initial partitions over all instances. The run-time difference gives the percentage change in the mean run time of the local search relative to using \\texttt{maximum\\_matching} as the initial partition. Positive values indicate a higher run time and negative values a lower run time for the alternative initial partition."
    ),
    label="tab:maximum_matching_pairwise",
)

print(pairwise_latex)

\begin{table}[H]
\centering
\caption{Pairwise comparison of \texttt{maximum\_matching} with the alternative initial partitions over all instances. The run-time difference gives the percentage change in the mean run time of the local search relative to using \texttt{maximum\_matching} as the initial partition. Positive values indicate a higher run time and negative values a lower run time for the alternative initial partition.}
\label{tab:maximum_matching_pairwise}
\begin{tabular}{p{4.8cm}rrrr}
\toprule
Initial partition & \shortstack{Mean relative\\solution quality} & \shortstack{Maximum\\Matching\\better} & \shortstack{Maximum\\Matching\\worse} & \shortstack{Run time\\difference (\%)} \\
\midrule
\texttt{singleton} & 1.004715 & 55.2\,\% & 36.0\,\% & -3.8\,\% \\
\texttt{matching} & 1.004580 & 52.9\,\% & 37.7\,\% & -5.5\,\% \\
\texttt{maximum\_matching} & \textbf{1.004227} & -- & -- & -- \\
\texttt{maximum\_matching\_edge\_cover} & 1.004244 & 21.0\,\% & 20.1\,\% & +0.9\,\% \\
\texttt{hi